# Quickstart: nuclei in 3D

Segments the nuclei in 2D first (as in `1_quickstart_2d.ipynb`) and then finds, around every 2D outline, the closed
envelope of the nucleus in the z-stack. For every stack it writes to `OUT_DIR` the 2D files and

- `<name>_labels_3d.tif`: a 3D label stack at the resolution of the input, labels as in the 2D table,
- `<name>_nuclei_3d.csv`: volume, centroid, height and z extent per nucleus,
- `<name>_sections_3d.png`: the outlines on every slice, to check the result.

The 3D step needs the slice distance, read from the file (ImageJ `spacing`, OME `PhysicalSizeZ`) or set as
`z_step_um`. The stage step often differs from the true distance in the sample (refractive-index mismatch);
`4_method_3d.ipynb` shows how volumes depend on it.

**Data.** Put your TIFF z-stacks in `data/` next to this notebook, or set `DATA_DIR` to their folder. Without
any, the notebook writes and uses a small synthetic stack (`example_data/synthetic.tif`), which shows how the pipeline
works but is no substitute for real images. See the README for the input the pipeline expects.

**Start with one stack** (about a minute each): uncomment `paths = paths[:1]` below, check the result and adjust the
parameters, then run all stacks.

In [ ]:
from pathlib import Path

from IPython.display import Image, display
import pandas as pd

import nucseg
from nucseg.surfaces import plot_3d

In [ ]:
DATA_DIR = Path("data")      # folder with your TIFF z-stacks
OUT_DIR = Path("results")
CHANNEL = None               # channel of multichannel files (0-based); None for single-channel files
UM_PER_PIXEL = None          # pixel size in µm; None reads it from the ImageJ/OME metadata

paths = nucseg.find_stacks([DATA_DIR]) if DATA_DIR.exists() else []
if not paths:
    print(f"No TIFFs in {DATA_DIR}/, using the synthetic example instead.")
    paths = [nucseg.write_synthetic_stack(Path("example_data") / "synthetic.tif")]
# paths = paths[:1]          # try one stack first
[p.name for p in paths]

## Parameters

As in `1_quickstart_2d.ipynb`:

| | |
|---|---|
| **read from the file** | pixel size and z step (ImageJ/OME metadata), printed for every stack; set `UM_PER_PIXEL` above if it is missing or wrong |
| **automatic** | intensity scaling (each projection is scaled to its own 99.8th percentile), the projection each nucleus is taken from |
| **set for your data** | `window`: slices averaged per projection, about 1 µm (3 at 0.3 µm slices) |
| | `min_area_um2`, `max_area_um2`, `min_width_um`: the size of your nuclei |
| | `border_frame`: also find the visible part of nuclei cut by the image edge |
| **speed** | `bin_factor`: pixels averaged per axis; higher is faster, but too coarse loses thin rings |
| | `z_first`, `z_last`: the slices through the nuclei (default: all slices) |
| **tuning only** | everything else; see the table *Tuning* at the end of `2_method_2d.ipynb` |

The 3D step uses the defaults of `nucseg.Parameters3D` (see `help(nucseg.Parameters3D)`); the one to set is `z_step_um` if the slice distance in the file is missing or not the true distance in the sample.

In [ ]:
parameters = nucseg.Parameters(
    bin_factor=4,          # pixels averaged per axis, for speed; too coarse loses thin rings
    window=3,              # slices averaged per projection, about 1 µm
    min_area_um2=25.0,     # size of your nuclei
    max_area_um2=800.0,
    min_width_um=5.0,
    z_first=None,          # slices for the 2D outlines (inclusive); None = first / last
    z_last=None,
    border_frame=True,     # also nuclei cut by the image edge, marked border=True (default False)
)
parameters_3d = nucseg.Parameters3D(
    z_step_um=None,        # slice distance in µm; None reads it from the file
)

## Segmentation

About 1 min per 2048 x 2048 x 20 stack (Apple M4 Pro): 35-40 s for 2D, 20-30 s for 3D; less with a narrower z range (`z_first`, `z_last`).

Check the outlines on every slice, and the overview and gallery of the 2D step in `OUT_DIR`. If nuclei are
missing, split or merged, or outlines run into the ER, the table *Tuning* at the end of `2_method_2d.ipynb` lists the
parameter to change for each symptom.

In [ ]:
tables, results = [], {}
for path in paths:
    stack = nucseg.read_stack(path, channel=CHANNEL, um_per_pixel=UM_PER_PIXEL)
    z_step = "unknown" if stack.z_step_um is None else f"{stack.z_step_um:.3f} µm"
    print(f"{stack.name}: {stack.data.shape[0]} slices of {stack.data.shape[1]} x {stack.data.shape[2]} px, "
          f"{stack.um_per_pixel:.4f} µm/px, z step {z_step}; segmenting ...")
    segmentation = nucseg.segment_2d(stack, parameters)
    segmentation_3d = nucseg.segment_3d(stack, segmentation, parameters_3d)
    nucseg.save_results(OUT_DIR, stack, segmentation, segmentation_3d)
    results[stack.name] = segmentation_3d
    tables.append(pd.DataFrame(segmentation_3d.table()).assign(image=stack.name))
    print(f"  {len(segmentation.nuclei)} nuclei in 2D, "
          f"{sum(n.found for n in segmentation_3d.nuclei)} in 3D")
    display(Image(OUT_DIR / f"{stack.name}_sections_3d.png"))

## Nuclei

One row per 2D nucleus (`found = False` if no closed envelope was found around it). If `top_cap` or `bottom_cap` is
set, the nucleus reaches the first or last slice and is closed there artificially: its height and volume are lower
bounds. For `border = True` they are those of the visible part.

In [ ]:
nuclei = pd.concat(tables, ignore_index=True).set_index(["image", "label"])
nuclei

## 3D view

Rotate with the mouse. `z_exaggeration` stretches z for viewing.

In [ ]:
name = next(iter(results))
fig = plot_3d(results[name], z_exaggeration=2.0)
fig.update_layout(height=700, title=name)
fig.write_html(OUT_DIR / f"{name}_3d.html", include_plotlyjs=True)
fig.show()